# MDP Lesson 5: finite-horizon MDP

This C++ notebook is the notebook form of **MDP Example 40**. It implements the finite-horizon stochastic inventory model described by Puterman and solves it with the Marmote finite-horizon MDP API.

**Import the modules**

In [1]:
// --- Marmote configuration for Xeus-cling ---
// These directives are technical and hidden from the rendered documentation.
#ifdef _WIN32
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteCore")
#pragma cling add_include_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/include/marmoteMDP")
#pragma cling add_library_path("C:/Users/assia/miniconda3/envs/jcpp-win/Library/bin")
#pragma cling load("marmoteCore.dll")
#pragma cling load("marmoteMDP.dll")
#else
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteCore")
#pragma cling add_include_path("/home/assia/miniconda3/envs/xeus-cpp-env/include/marmoteMDP")
#pragma cling add_library_path("/home/assia/miniconda3/envs/xeus-cpp-env/lib")
#pragma cling load("libmarmoteCore.so")
#pragma cling load("libmarmoteMDP.so")
#endif


The notebook uses the core Marmote classes for sets and matrices, together with the finite-horizon MDP classes.

In [2]:
// --- Standard C++ utilities used in this notebook ---
#include <iostream>
#include <string>
#include <vector>

// --- Marmote headers used in this lesson ---
#include <marmoteCore/marmoteFullMatrix.h>
#include <marmoteCore/marmoteInterval.h>
#include <marmoteCore/marmoteSet.h>
#include <marmoteCore/marmoteSparseMatrix.h>
#include <marmoteMDP/marmoteFiniteHorizonMDP.h>
#include <marmoteMDP/marmoteNonStationarySolutionMDP.h>
#include <marmoteMDP/marmoteSolutionMDP.h>

// --- Convenience declarations for the cells below ---
using namespace std;
using namespace marmote;


## Description of the model

We consider a finite-horizon inventory problem with state space `[0,3]`, action space `[0,3]`, horizon `N = 3` and optimization criterion `max`. The transition matrices follow the data of Puterman's example.

In [3]:
// Define the main features of the finite-horizon MDP.
string criterion = "max";
double penalty = -1000.0;
int horizon = 3;
double discountFactor = 1.0;

// The solver signature still requires epsilon and maxIter, even though they
// do not drive a standard fixed-point iteration in the finite-horizon case.
double epsilon = 0.01;
int maxIter = 7;


## Create the state space, action space and transition matrices

In [4]:
// Create the one-dimensional state and action spaces.
int minState = 0;
int maxState = 3;
MarmoteSet* actionSpace = new MarmoteInterval(0, 3);
MarmoteSet* stateSpace = new MarmoteInterval(minState, maxState);

// Prepare the collection of transition matrices, one for each action.
vector<TransitionStructure*> trans(actionSpace->Cardinal());
SparseMatrix* P = nullptr;


In [5]:
// Transition matrix for action 0.
P = new SparseMatrix(stateSpace->Cardinal());
P->setEntry(0, 0, 1.0);
P->setEntry(1, 0, 0.75);
P->setEntry(1, 1, 0.25);
P->setEntry(2, 0, 0.25);
P->setEntry(2, 1, 0.5);
P->setEntry(2, 2, 0.25);
P->setEntry(3, 1, 0.25);
P->setEntry(3, 2, 0.5);
P->setEntry(3, 3, 0.25);
trans.at(0) = P;
cout << "Added transition matrix 0 to vector" << endl;

// Transition matrix for action 1.
P = new SparseMatrix(stateSpace->Cardinal());
P->setEntry(0, 0, 0.75);
P->setEntry(0, 1, 0.25);
P->setEntry(1, 0, 0.25);
P->setEntry(1, 1, 0.5);
P->setEntry(1, 2, 0.25);
P->setEntry(2, 1, 0.25);
P->setEntry(2, 2, 0.5);
P->setEntry(2, 3, 0.25);
P->setEntry(3, 1, 0.25);
P->setEntry(3, 2, 0.5);
P->setEntry(3, 3, 0.25);
trans.at(1) = P;
cout << "Added transition matrix 1 to vector" << endl;

// Transition matrix for action 2.
P = new SparseMatrix(stateSpace->Cardinal());
P->setEntry(0, 0, 0.25);
P->setEntry(0, 1, 0.5);
P->setEntry(0, 2, 0.25);
P->setEntry(1, 1, 0.25);
P->setEntry(1, 2, 0.5);
P->setEntry(1, 3, 0.25);
P->setEntry(2, 1, 0.25);
P->setEntry(2, 2, 0.5);
P->setEntry(2, 3, 0.25);
P->setEntry(3, 1, 0.25);
P->setEntry(3, 2, 0.5);
P->setEntry(3, 3, 0.25);
trans.at(2) = P;
cout << "Added transition matrix 2 to vector" << endl;

// Transition matrix for action 3.
P = new SparseMatrix(stateSpace->Cardinal());
P->setEntry(0, 1, 0.25);
P->setEntry(0, 2, 0.5);
P->setEntry(0, 3, 0.25);
P->setEntry(1, 1, 0.25);
P->setEntry(1, 2, 0.5);
P->setEntry(1, 3, 0.25);
P->setEntry(2, 1, 0.25);
P->setEntry(2, 2, 0.5);
P->setEntry(2, 3, 0.25);
P->setEntry(3, 1, 0.25);
P->setEntry(3, 2, 0.5);
P->setEntry(3, 3, 0.25);
trans.at(3) = P;
cout << "Added transition matrix 3 to vector" << endl;


Added transition matrix 0 to vector
Added transition matrix 1 to vector
Added transition matrix 2 to vector
Added transition matrix 3 to vector


## Build the reward matrix and the finite-horizon MDP

In [ ]:
// Build the reward matrix.
FullMatrix* R = new FullMatrix(stateSpace->Cardinal(), actionSpace->Cardinal());
R->setEntry(0, 0, 0);
R->setEntry(0, 1, -1);
R->setEntry(0, 2, -2);
R->setEntry(0, 3, -5);
R->setEntry(1, 0, 5);
R->setEntry(1, 1, 0);
R->setEntry(1, 2, -3);
R->setEntry(1, 3, penalty);
R->setEntry(2, 0, 6);
R->setEntry(2, 1, -1);
R->setEntry(2, 2, penalty);
R->setEntry(2, 3, penalty);
R->setEntry(3, 0, 5);
R->setEntry(3, 1, penalty);
R->setEntry(3, 2, penalty);
R->setEntry(3, 3, penalty);

// Build the finite-horizon MDP.
cout << "Beginning building MDP" << endl;
FiniteHorizonMDP* mdp1 = new FiniteHorizonMDP(criterion, stateSpace, actionSpace, trans, R, horizon, discountFactor);
cout << "MDP built" << endl;
mdp1->Write();


## Solve the MDP

The finite-horizon solver returns a **non-stationary** solution, which is written step by step.

In [7]:
// Solve the model by dynamic programming.
cout << "Print solution after value iteration" << endl;
NonStationarySolutionMDP* optimum = mdp1->ValueIteration(epsilon, maxIter);
optimum->Write();


Print solution after value iteration
#############################################
Solution of MDP problem
Size of the state space: 4
# Horizon: 3
#############################################
# Solution of the entered problem model:
# - column 1: index of the state 
# - column 2: Value function 
# - column 3: Optimal action 
#
Step 0
   0   4.1875   3
   1   8.0625   0
   2   12.125   0
   3   14.188   0
####
Step 1
   0   2   2
   1   6.25   0
   2   10   0
   3   10.5   0
####
Step 2
   0   0   0
   1   5   0
   2   6   0
   3   5   0
####
#############################################



## Analyse the solution

We evaluate the policy again and print the value at the last decision step for each state.

In [8]:
// Evaluate the computed policy and inspect the values at step horizon - 1.
cout << endl << "Checking solutions" << endl;
mdp1->PolicyCost(optimum, epsilon, maxIter);
for (stateType i = 0; i < stateSpace->Cardinal(); i++) {
    cout << "i = " << i << " N = " << horizon - 1
         << " solution = " << optimum->getValueAtStepIndex(horizon - 1, i) << endl;
}



Checking solutions
i = 0 N = 2 solution = 0
i = 1 N = 2 solution = 5
i = 2 N = 2 solution = 6
i = 3 N = 2 solution = 5


In [9]:
// Release the MDP objects created in this lesson.
delete optimum;
delete mdp1;
delete stateSpace;
delete actionSpace;
